# MiniLAr + CNN

In this notebook, we design and train a Convolutional Neural Network (CNN) for hand-written digit classification task. We use MNIST dataset that contains 28x28 pixel images of a hand-written digit (0 to 9, so 10 classification targets). 

## Goals
1. Design CNN and train on MNIST

Let's start with usual import!

In [ ]:
# Import drawing tools, set style
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

mpl.rcParams['figure.figsize'] = [8, 6]
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.grid'] = True

# Import numpy and torch, set default device based on GPU availability
import numpy as np
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.device(device)

# Set the seed to always get the same result from RNG calls in this notebook
SEED = 12345
np.random.seed(SEED)    # Setting the seed for reproducibility
torch.manual_seed(SEED) # This is how you do for torch!

## Create MNIST Dataset
Following the previous notebook, let's create train and test dataset and dataloader

In [ ]:
from dataset.lartpc_particles import LArTPCParticles

# Specify where you want the dataset to land
LOCAL_DATA_DIR = 'dataset/mlar-dataset'

# Fill this in once the dataset file is hosted.
# You can also use a local path, e.g. DATA_URL='/path/to/lartpc_particles.pt'
DATA_URL = 'https://s3df.slac.stanford.edu/data/neutrino/dune-tech-analysis/mlar/raw/lartpc_particles.pt'

# Use prepared data handler from pytorch (torchvision)
train_dataset = LArTPCParticles(
    LOCAL_DATA_DIR,
    train=True,
    download=DATA_URL is not None,
    url=DATA_URL,
)

train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=64,
                                           shuffle=True,
                                           num_workers=4)

# Use prepared data handler from pytorch (torchvision)
test_dataset = LArTPCParticles(
    LOCAL_DATA_DIR,
    train=False,
    download=DATA_URL is not None,
    url=DATA_URL,
)

test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=64,
                                          shuffle=False,
                                          num_workers=4)

### Define train and test functions

We recycle those from the previous MLP notebook!

In [ ]:
# Python time tracking package
import time

# Python CSV writing package
import csv

# Progress bar package, makes waiting easier!
# If you're curious, tqdm comes from the arabic word for progress, taqadum
from tqdm import tqdm

def run_train(model, loader,  
              num_iterations=100, log_dir='logs_mlp', log_prefix=None,
              optimizer='SGD', lr=0.001, device=None):
    """Train loop function.
    
    Parameters
    ----------
    model : torch.nn.Module
        Machine learning model
    loader : torch.DataLoader
        Batch loading tool
    num_iterations : int, default 100
        Number of iterations to run the optimization over
    log_dir : str, default 'logs_mlp'
        Path to the log directory
    log_prefix : str, optional
        Prefix to prepend onto log names
    optimizer : str, default 'SGD'
        Name of the optimizer as defined here: https://docs.pytorch.org/docs/stable/optim.html
    lr : float, default 0.001
        Learning rate
    """
    # Create the log directory if it does not exist
    !mkdir -p {log_dir}

    # Define log name
    prefix = '' if log_prefix is None else f'{log_prefix}_'
    train_log_name = f'{log_dir}/{prefix}train.csv'

    # Initialize the loss function (cross-entropy loss)
    criterion = torch.nn.CrossEntropyLoss()

    # Initialize the optimizer
    optimizer_fn = getattr(torch.optim,optimizer)
    optimizer = optimizer_fn(model.parameters(), lr=lr)

    # Initialize a progress bar
    pbar = tqdm(total=num_iterations, position=0, leave=True)

    # Loop over the entire dataset until the requested number of iterations is reached
    iteration = 0
    with open(train_log_name, 'w') as csvfile:
        # Initialize CSV writer
        writer = csv.writer(csvfile)
        writer.writerow(['iter', 'epoch', 'loss'])
        while iteration < num_iterations:
            # Loop over data in the loader (one epoch)
            for data, label in loader:
                # If a device is specify, move data/label to it
                if device is not None:
                    data, label = data.to(device), label.to(device)

                optimizer.zero_grad()
    
                loss = criterion(model(data), label)
                
                loss.backward()
                optimizer.step()
                
                # Append the log
                writer.writerow([iteration, iteration/len(loader), loss.item()])
    
                # Increment iteration count, update the progress bar
                iteration += 1
                pbar.update(1)

                # Break if the necessary number of iterations is reached
                if iteration >= num_iterations:
                    break


def run_test(model, loader, device=None):
    """Runs the trained model on test data.

    Parameters
    ----------
    model : torch.nn.Module
        Model definition
    loader : torch.DataLoader
        Method to load the test data
    device : str, optional
        Device on which to put the data/model if one uses a GPU

    Returns
    -------
    torch.Tensor
        (N) List of labels for each of the image in the test test
    torch.Tensor
        (N, 10) Softmax scores predicted for each of the 10 possible digits
    """
    # Initialize the output lists
    label_v, softmax_v = [],[]

    # Initialize the softmax function, s_i = exp(-v_i)/(sum_i exp(-v_i))
    softmax   = torch.nn.Softmax(dim=1)

    # Initialize a progress bar
    pbar = tqdm(total=len(loader), position=0, leave=True)

    # Loop over test data, append labels and softmax predictions
    with torch.set_grad_enabled(False):
        for data, label in loader:
            if device is not None:
                data,label = data.to(device), label.to(device)
            label_v.append  ( label.detach().reshape(-1)   )
            softmax_v.append( softmax(model(data)).detach())
            pbar.update(1)

    # Return
    return torch.concat(label_v).cpu().numpy(), torch.concat(softmax_v).cpu().numpy()

## Logistic regression with CNN

we design CNN to try the same task. Let's define 3 convolution layers followed by LeakyReLU for activation and MaxPool2d for downsampling.

In [ ]:
NUM_CLASSES = 5

class CNN(torch.nn.Module):
    """Basic convolution neural network (CNN) model."""

    def __init__(self, num_filters=16):
        """Initialize the CNN.

        Parameters
        ----------
        num_filters : int, default 16
            Number of filter kernels (convolution matrices)
        """
        # Initialize the parent class
        super().__init__()
        
        # CNN feature extractor definition
        self._feature_extractor = torch.nn.Sequential(
            torch.nn.Conv2d(1, num_filters, 3, padding=1),             # 1 to N_f conv, kernel size 3, padding 1 (preserve size)
            torch.nn.LeakyReLU(),                                      # LeakyReLU activation
            torch.nn.MaxPool2d(2, 2),                                  # Max pooling, now image is 14*14
            torch.nn.Conv2d(num_filters, num_filters*2, 3, padding=1), # N_f to 2*N_f conv, kernel size 3, padding 1
            torch.nn.LeakyReLU(),                                      # LeakyReLU activation
            torch.nn.MaxPool2d(2, 2),                                  # Max pooling, now image is 7*7
            torch.nn.Conv2d(num_filters*2,num_filters*4,3,padding=1),  # 2*N_f to 4*N_f conv, kernel size 3, padding 1
            torch.nn.LeakyReLU(),                                      # LeakyReLU activation
            torch.nn.MaxPool2d(7, 7))                                  # Final max pooling, pools everything into a single feature vector
        
        # Classifier MLP, converts the feature vector (4*N_f) into a 10 values (one per possible class)
        self._classifier = torch.nn.Linear(num_filters*4, NUM_CLASSES)

    def forward(self, x):
        """Pass one batch of data through the model.

        Parameters
        ----------
        x : torch.Tensor
            (N, 1, 28, 28) N digit images of size 28*28

        Returns
        -------
        torch.Tensor
            (N) Logit output of the classifier layer (one number per image)
        """
        
        # Extract features
        features = self._feature_extractor(x)
        
        # Flatten the 3d tensor (2d space x channels = features)
        features = features.view(-1, np.prod(features.size()[1:]))
        
        # Classify and return
        return self._classifier(features)

### Train the model
Let's use the train loop to train the model

In [ ]:
# Set the device
device = 0 # Change to to GPU index to run on GPU, typically 0

# Initialize an CNN with 16 filters (place on the correct device!)
model = CNN(16).to(device=device)

# Run the train loop
run_train(model, train_loader, 10000, log_dir='logs_cnn', optimizer='Adam', device=device)

If your machine is slow, skip over this training process and simply load the model parameters from an earlier training process!

The way this file was produced was by simply calling
```python
torch.save(model.state_dict(), 'weights/cnn_model.ckpt')
```

In [ ]:
torch.save(model.state_dict(), 'weights/cnn_model.ckpt')

In [ ]:
state_dict = torch.load('weights/cnn_model.ckpt', weights_only=True)
model.load_state_dict(state_dict)

Now we look at the train log, we use the one trained earlier to save time...

In [ ]:
import pandas as pd

df = pd.read_csv('logs_cnn/train.csv')
# df = pd.read_csv('logs_cnn/pretrain.csv')
df

In [ ]:
df.rolling(10).mean().plot('epoch', 'loss', grid=True)

You can see that this model converges to a lower loss value than the MLP!

Now let us evaluate exactly how much better it does on a test set...

In [ ]:
labels, scores = run_test(model, test_loader, device=device)

Once again, if your machine is slow, I have done this for you upstream, simply load the information!

In [ ]:
# saved_data = torch.load('inference/cnn_output.pkl', weights_only=False)
# labels, scores = saved_data['labels'], saved_data['scores']

Now let's check accuracy:

In [ ]:
preds = np.argmax(scores, axis=1)
np.sum(labels == preds)/len(labels)

98.4 %, that's a large improvement w.r.t. the MLP!

aaaaaand a confusion matrix:

In [ ]:
import plotly.figure_factory as ff

# Define a function which estimates the accuracy of the model
def make_confusion_matrix(labels, preds):
    """Make a confusion matrix.

    Parameters
    ----------
    labels : torch.Tensor
        Labels associated with each image
    preds : torch.Tensor
        Predictions from the network associated with each image

    Returns
    -------
    np.ndarray
        (NUM_CLASSES, NUM_CLASSES) confusion matrix
    """
    # Loop over possible labels/prediction pair and count the numbers
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=float)
    for l, p in zip(labels, preds):
        cm[l, p] += 1

    # Normalize to the number of labels in each category
    for i in range(NUM_CLASSES):
        cm[i, :] /= np.sum(cm[i])

    # Return
    return cm

# Get the confusion matrix for the test set
cm = make_confusion_matrix(labels, preds)

# Convert the confusion matrix to a set of strings (to draw)
cm_txt = np.round(cm, 2).astype(str)

# Initialize heatmap 
classes = list(train_dataset.class_names)
fig = ff.create_annotated_heatmap(z=cm, x=classes, y=classes, annotation_text=cm_txt, colorscale='Viridis')

# Define the heatmat layout
fig.update_layout(
    margin=dict(l=20, r=20, t=20, b=20), # Margins around the plot
    xaxis_title='Prediction', yaxis_title='True label', # Axis titles
    width=500, height=500 # Dimensions of the plot
)

# Draw!
fig.show()

### Exercise 1

Train the CNN model on GPU, compare the amount of time taken to do it!

In [ ]:
# Your code here!

You should see that, this time, GPU gave a good speed-up. This is because CNN takes many separate multiplications of weights with input local matrix, and that can benefit from parallelization = GPU is suited.

### Exercise 2
How many parameters are there in our CNN model?

In [ ]:
# Your code here!

In [ ]:
torch.save(model.state_dict(), 'weights/cnn_model.ckpt')